In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("spam (1).csv", encoding='cp1252')

df = df[['v1','v2']]

df.columns = ['label','message']

print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [3]:
df['label'] = df['label'].map({
    'ham':0,
    'spam':1
})

In [5]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')

ps = PorterStemmer()

def clean_text(text):

    text = re.sub('[^a-zA-Z]', ' ', text)

    text = text.lower()

    words = text.split()

    words = [
        ps.stem(word)
        for word in words
        if word not in stopwords.words('english')
    ]

    return " ".join(words)

df['message'] = df['message'].apply(clean_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['message']).toarray()

y = df['label']

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb = MultinomialNB()

nb.fit(X_train,y_train)

pred_nb = nb.predict(X_test)

print("Naive Bayes Accuracy:",
      accuracy_score(y_test,pred_nb))

Naive Bayes Accuracy: 0.968609865470852


In [9]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()

lr.fit(X_train,y_train)

pred_lr = lr.predict(X_test)

print("Logistic Regression Accuracy:",
      accuracy_score(y_test,pred_lr))

Logistic Regression Accuracy: 0.95695067264574


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train,y_train)

pred_rf = rf.predict(X_test)

print("Random Forest Accuracy:",
      accuracy_score(y_test,pred_rf))

Random Forest Accuracy: 0.979372197309417


In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

I0000 00:00:1780989603.809820   15346 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780989605.657793   15346 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780989608.755112   15346 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [16]:
model = Sequential()

model.add(Dense(
    128,
    activation='relu',
    input_shape=(5000,)
))

model.add(Dense(
    64,
    activation='relu'
))

model.add(Dense(
    1,
    activation='sigmoid'
))

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1780989612.305098   15346 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [17]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [18]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5


W0000 00:00:1780989628.470998   15346 cpu_allocator_impl.cc:82] Allocation of 71300000 exceeds 10% of free system memory.


112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8808 - loss: 0.3200 - val_accuracy: 0.9765 - val_loss: 0.1379
Epoch 2/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9882 - loss: 0.0559 - val_accuracy: 0.9765 - val_loss: 0.0726
Epoch 3/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9969 - loss: 0.0143 - val_accuracy: 0.9787 - val_loss: 0.0749
Epoch 4/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9989 - loss: 0.0066 - val_accuracy: 0.9776 - val_loss: 0.0774
Epoch 5/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9992 - loss: 0.0038 - val_accuracy: 0.9753 - val_loss: 0.0874


In [19]:
loss, acc = model.evaluate(
    X_test,
    y_test
)

print("Deep Learning Accuracy:", acc)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9776 - loss: 0.0864
Deep Learning Accuracy: 0.9775784611701965


W0000 00:00:1780989636.209418   15346 cpu_allocator_impl.cc:82] Allocation of 22300000 exceeds 10% of free system memory.


In [20]:
print("\nModel Comparison")

print("Naive Bayes:",
      accuracy_score(y_test,pred_nb))

print("Logistic Regression:",
      accuracy_score(y_test,pred_lr))

print("Random Forest:",
      accuracy_score(y_test,pred_rf))

print("Neural Network:",
      acc)


Model Comparison
Naive Bayes: 0.968609865470852
Logistic Regression: 0.95695067264574
Random Forest: 0.979372197309417
Neural Network: 0.9775784611701965


In [21]:
import pickle

with open("phishing_model.pkl", "wb") as file1:
    pickle.dump(model, file1)

print("Model saved successfully!")

Model saved successfully!


In [22]:
with open("vectorizer.pkl", "wb") as file2:
    pickle.dump(tfidf, file2)

print("Vectorizer saved successfully!")

Vectorizer saved successfully!


In [23]:
import os
print(os.listdir())

['vectorizer.pkl', 'phishing_model.pkl', 'project.ipynb', 'spam (1).csv']


In [24]:
with open("phishing_model.pkl", "rb") as file1:
    loaded_model = pickle.load(file1)

with open("vectorizer.pkl", "rb") as file2:
    loaded_vectorizer = pickle.load(file2)

print("Files loaded successfully!")

Files loaded successfully!


In [29]:
message = ["Congratulations! You won a free iPhone. Click now"]

message_vector = loaded_vectorizer.transform(message)

# Get probability
probability = loaded_model.predict(message_vector)

spam_probability = float(probability[0][0]) * 100

# Convert probability to prediction
prediction = 1 if spam_probability >= 50 else 0

print("Prediction:", prediction)
print(f"Spam Probability: {spam_probability:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
Prediction: 1
Spam Probability: 64.57%


In [33]:
from tensorflow.keras.models import load_model
loaded_model = load_model("phishing_model.keras")

ValueError: File not found: filepath=phishing_model.keras. Please ensure the file is an accessible `.keras` zip file.